## METHOD 2: Neural Network + AdamW Optimizer

In [ ]:
%pip install pandas numpy
import pandas as pd
import numpy as np

gain_df = pd.read_csv(r"/content/gain_clean.csv")
ugf_df  = pd.read_csv(r"/content/ugf_clean.csv")
pm_df   = pd.read_csv(r"/content/pm_clean.csv")

X = gain_df[["a", "b", "c", "d"]].values

Y = np.column_stack([
    gain_df["gain"].values,
    ugf_df["ugf"].values,
    pm_df["pm"].values
])


from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import joblib

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

kernel = ConstantKernel(1.0) * RBF(1.0)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=5
)

model = MultiOutputRegressor(gpr)
model.fit(X_train, Y_train)

preds = model.predict(X_test)

print("R2 Gain:", r2_score(Y_test[:,0], preds[:,0]))
print("R2 UGF :", r2_score(Y_test[:,1], preds[:,1]))
print("R2 PM  :", r2_score(Y_test[:,2], preds[:,2]))

joblib.dump(model, "gpr_multioutput.pkl")



/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/_gpr.py:660: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
